In [14]:
import pandas as pd

# custom libs
import sys
sys.path.append('../../')
from src.preprocessing import df_to_densities

In [16]:
# Returns data
data_path = "../../data/processed/"
returns_path = ''.join([data_path, 'ibovespa_treated.xlsx'])
df_log_returns = pd.read_excel(returns_path, index_col="time")
df_log_returns.head()

,2024-12-02,2024-12-03,2024-12-04,2024-12-05,2024-12-06,2024-12-09,2024-12-10,2024-12-11,2024-12-12,2024-12-13,...,2025-11-14,2025-11-17,2025-11-18,2025-11-19,2025-11-21,2025-11-24,2025-11-25,2025-11-26,2025-11-27,2025-11-28
time,,,,,,,,,,,,,,,,,,,,,
10:05:00,-0.002271,0.003372,0.001716,0.005538,0.000069,0.006429,0.002343,0.001918,-0.007582,-0.000413,...,-0.001522,0.000739,-0.000613,0.000306,-0.002021,-0.000810,0.001557,-0.001565,-0.000193,0.002906
10:10:00,-0.000582,0.002832,-0.000502,0.000739,0.000261,0.001891,0.000775,-0.001172,0.000216,-0.000432,...,0.000360,-0.000105,-0.000159,-0.000362,0.000092,0.001131,0.001170,0.000202,0.000556,0.000475
10:15:00,-0.001997,0.001091,0.000111,-0.000192,0.000407,-0.001392,-0.000477,-0.001526,0.000198,-0.001821,...,0.001121,-0.000284,0.003272,-0.001107,-0.001012,-0.000174,0.001002,0.001815,-0.000168,0.000973
10:20:00,-0.000037,-0.000505,-0.000327,-0.000580,-0.000436,-0.000405,0.001263,0.000676,-0.002667,0.001033,...,0.000449,0.000335,0.000853,0.000981,-0.000057,-0.000096,0.000832,0.001951,0.001060,-0.001004
10:25:00,0.000364,-0.001532,0.000429,-0.000426,-0.000162,-0.000130,0.000060,0.000818,-0.000970,0.000670,...,-0.000871,-0.000559,0.001030,-0.000749,-0.001194,0.001338,0.000128,0.001279,-0.000162,-0.000651


In [ ]:
params = {"kernel": "t_student", "bandwidth": "scott", "adaptive": False}
grid, densities = df_to_densities(df_log_returns, params)

In [55]:
import numpy as np
import plotly.graph_objects as go

# Axes
x = densities.columns.astype(str)   # time
y = densities.index.values           # support
z = densities.values                 # density

fig = go.Figure(
    data=go.Surface(
        x=x,
        y=y,
        z=z,
        colorscale="OrRd",        # perceptually uniform
        # colorscale="Oryel",
        # colorscale="Tealgrn", 
        showscale=True,
        colorbar=dict(
            title="Density",
            # titleside="right",
            len=0.6,
            thickness=15
        ),
        lighting=dict(
            ambient=0.6,
            diffuse=0.8,
            roughness=0.4,
            specular=0.2
        ),
        lightposition=dict(
            x=100,
            y=200,
            z=300
        )
    )
)

fig.update_layout(
    title=dict(
        # text="KDE over time",
        x=0.5,
        xanchor="center"
    ),
    scene=dict(
        xaxis=dict(
            title="Time",
            showgrid=False,
            showbackground=False,
            tickangle=45
        ),
        yaxis=dict(
            title="Support",
            showgrid=False,
            showbackground=False,
            visible=False
        ),
        zaxis=dict(
            title="Density",
            showgrid=False,
            showbackground=False,
            visible=False
        ),
        camera=dict(
            eye=dict(x=1.6, y=-1.6, z=0.9)
        ),
        aspectratio=dict(x=2.2, y=1.0, z=0.6)
    ),
    template="plotly_white",
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.update_layout(
scene_camera=dict(
eye=dict(x=1.8, y=1.8, z=0.6)
)
)

fig.update_traces(
contours_z=dict(
show=True,
project_z=True,
usecolormap=True
)
)

# Mask zero and non-zero values
z_nonzero = np.where(z > 0, z, np.nan)
eps = 0.00
z_zero = np.where(z <= eps, 0, np.nan)
# Gray surface for zeros
fig.add_trace(
go.Surface(
x=x,
y=y,
z=z_zero,
colorscale=[[0, "lightgray"], [1, "lightgray"]],
showscale=False,
opacity=1.0,
name="Zero density"
)
)

fig.update_layout(
width=900,
height=500
)

fig.show()

In [36]:
bovespa_mLQDT = mLQDT(
                    densities.reset_index(drop=True),
                    grid.reset_index(drop=True)
                )
bovespa_mLQDT.densities_to_lqdensities(verbose=False)

df_lqds = bovespa_mLQDT.lqd.iloc[1:-1,:]
df_lqds_support = bovespa_mLQDT.lqd_support[1:-1]

In [54]:
import numpy as np
import plotly.graph_objects as go

# Axes
x = df_lqds.columns.astype(str)   # time
y = df_lqds_support           # support
z = df_lqds.values                 # density

fig = go.Figure(
    data=go.Surface(
        x=x,
        y=y,
        z=z,
        # colorscale="OrRd",        # perceptually uniform
        # colorscale="Oryel",
        colorscale="OrRd", 
        showscale=True,
        colorbar=dict(
            title="Density",
            # titleside="right",
            len=0.6,
            thickness=15
        ),
        lighting=dict(
            ambient=0.6,
            diffuse=0.8,
            roughness=0.4,
            specular=0.2
        ),
        lightposition=dict(
            x=100,
            y=200,
            z=300
        )
    )
)

fig.update_layout(
    title=dict(
        # text="KDE over time",
        x=0.5,
        xanchor="center"
    ),
    scene=dict(
        xaxis=dict(
            title="Time",
            showgrid=False,
            showbackground=False,
            tickangle=45
        ),
        yaxis=dict(
            title="Support",
            showgrid=False,
            showbackground=False,
            visible=False
        ),
        zaxis=dict(
            title="Density",
            showgrid=False,
            showbackground=False,
            visible=False
        ),
        camera=dict(
            eye=dict(x=1.6, y=-1.6, z=0.9)
        ),
        aspectratio=dict(x=2.2, y=1.0, z=0.6)
    ),
    template="plotly_white",
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.update_layout(
scene_camera=dict(
eye=dict(x=1.8, y=1.8, z=0.6)
)
)

fig.update_traces(
contours_z=dict(
show=True,
project_z=True,
usecolormap=True
)
)

# Mask zero and non-zero values
z_nonzero = np.where(z > 0, z, np.nan)
z_zero = np.where(z == 0, 0, np.nan)
# Gray surface for zeros
fig.add_trace(
go.Surface(
x=x,
y=y,
z=z_zero,
colorscale=[[0, "lightgray"], [1, "lightgray"]],
showscale=False,
opacity=1.0,
name="Zero density"
)
)

fig.update_layout(
width=900,
height=500
)

fig.show()